# SQL-to-Redis Query Translation: Hands-On Exercises

This notebook provides hands-on exercises for learning the new **SQLQuery** feature in RedisVL, which allows you to write familiar SQL syntax that automatically translates to Redis Search commands.

## What You'll Learn

1. How to use the `SQLQuery` class to write SQL-like queries
2. Three equivalent approaches for the same queries:
   - **RedisVL Python API** - Using native query classes (`FilterQuery`, `VectorQuery`, etc.)
   - **RedisVL SQL** - Using the new `SQLQuery` class with SQL syntax
   - **Raw Redis FT.SEARCH** - The equivalent Redis Search command
3. Various query types: filtering, numeric ranges, text search, aggregations, and vector similarity

## Prerequisites

- Redis Stack running locally (or Redis Cloud)
- RedisVL with SQL support: `pip install redisvl[sql-redis]`

## Documentation References

- [RedisVL Documentation](https://docs.redisvl.com)
- [Redis Search Query Syntax](https://redis.io/docs/latest/develop/ai/search-and-query/query/)
- [Redis Aggregations](https://redis.io/docs/latest/develop/ai/search-and-query/advanced-concepts/aggregations/)
- [sql-redis Package](https://pypi.org/project/sql-redis/)

---

## Setup: Create Sample Dataset and Index

We'll create a realistic e-commerce products dataset with multiple field types to demonstrate various query capabilities.

In [1]:
import numpy as np
from redis import Redis
from redisvl.index import SearchIndex
from redisvl.query import FilterQuery, VectorQuery, CountQuery, SQLQuery
from redisvl.query.filter import Tag, Num, Text

# Redis connection
REDIS_URL = "redis://localhost:6379"
client = Redis.from_url(REDIS_URL)

# Define schema with multiple field types
schema = {
    "index": {
        "name": "products_exercise",
        "prefix": "product_exercise",
        "storage_type": "hash",
    },
    "fields": [
        {"name": "name", "type": "text", "attrs": {"sortable": True}},
        {"name": "description", "type": "text"},
        {"name": "category", "type": "tag", "attrs": {"sortable": True}},
        {"name": "brand", "type": "tag"},
        {"name": "price", "type": "numeric", "attrs": {"sortable": True}},
        {"name": "stock", "type": "numeric", "attrs": {"sortable": True}},
        {"name": "rating", "type": "numeric", "attrs": {"sortable": True}},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "dims": 4,
                "distance_metric": "cosine",
                "algorithm": "flat",
                "datatype": "float32",
            },
        },
    ],
}

# Create the index
index = SearchIndex.from_dict(schema, redis_client=client)
index.create(overwrite=True, drop=True)
print(f"Created index: {index.name}")

Created index: products_exercise


In [2]:
# Sample product data with embeddings
products = [
    {"name": "Gaming Laptop Pro", "description": "High-performance laptop for gaming", "category": "electronics", "brand": "TechBrand", "price": 1299, "stock": 15, "rating": 4.7, "embedding": np.array([0.9, 0.1, 0.2, 0.3], dtype=np.float32).tobytes()},
    {"name": "Budget Laptop Basic", "description": "Affordable laptop for everyday tasks", "category": "electronics", "brand": "ValueTech", "price": 499, "stock": 50, "rating": 4.0, "embedding": np.array([0.8, 0.2, 0.3, 0.4], dtype=np.float32).tobytes()},
    {"name": "Wireless Mouse", "description": "Ergonomic wireless mouse", "category": "electronics", "brand": "TechBrand", "price": 35, "stock": 200, "rating": 4.3, "embedding": np.array([0.7, 0.3, 0.4, 0.5], dtype=np.float32).tobytes()},
    {"name": "Python Programming Guide", "description": "Comprehensive Python programming guide", "category": "books", "brand": "TechBooks", "price": 45, "stock": 100, "rating": 4.8, "embedding": np.array([0.2, 0.8, 0.1, 0.3], dtype=np.float32).tobytes()},
    {"name": "Redis in Action", "description": "Learn Redis with practical examples", "category": "books", "brand": "TechBooks", "price": 55, "stock": 75, "rating": 4.6, "embedding": np.array([0.3, 0.7, 0.2, 0.4], dtype=np.float32).tobytes()},
    {"name": "Data Science Handbook", "description": "Essential data science handbook", "category": "books", "brand": "DataPress", "price": 65, "stock": 40, "rating": 4.5, "embedding": np.array([0.25, 0.75, 0.15, 0.35], dtype=np.float32).tobytes()},
    {"name": "Mechanical Keyboard", "description": "Premium mechanical keyboard with RGB", "category": "electronics", "brand": "KeyMaster", "price": 149, "stock": 80, "rating": 4.6, "embedding": np.array([0.6, 0.4, 0.5, 0.6], dtype=np.float32).tobytes()},
    {"name": "USB-C Hub", "description": "Multi-port USB-C hub", "category": "electronics", "brand": "TechBrand", "price": 49, "stock": 150, "rating": 4.2, "embedding": np.array([0.65, 0.35, 0.45, 0.55], dtype=np.float32).tobytes()},
    {"name": "Desk Lamp LED", "description": "Adjustable LED desk lamp", "category": "accessories", "brand": "LightCo", "price": 39, "stock": 120, "rating": 4.1, "embedding": np.array([0.4, 0.5, 0.6, 0.7], dtype=np.float32).tobytes()},
    {"name": "Monitor Stand", "description": "Ergonomic monitor stand", "category": "accessories", "brand": "DeskPro", "price": 79, "stock": 60, "rating": 4.4, "embedding": np.array([0.45, 0.55, 0.65, 0.75], dtype=np.float32).tobytes()},
]

# Load data into Redis
keys = index.load(products)
print(f"Loaded {len(keys)} products into Redis")

Loaded 10 products into Redis


---
## Exercise 1: Simple Tag Filtering

**Goal:** Find all products in the "electronics" category.

### Do It Yourself

**Documentation:**
- [RedisVL FilterQuery](https://docs.redisvl.com/en/latest/api/query.html#filterquery)
- [Redis Tag Queries](https://redis.io/docs/latest/develop/ai/search-and-query/query/exact-match/)

In [5]:
# YOUR CODE HERE - Method 1: RedisVL Python API
# Hint: Use Tag("category") == "electronics" with FilterQuery
q= FilterQuery(
    filter_expression=Tag("category") == "electronics",
    return_fields=["name", "category", "price"],
    num_results=10
)

q = index.query(q)
q

[{'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GK',
  'name': 'Wireless Mouse',
  'category': 'electronics',
  'price': '35'},
 {'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GQ',
  'name': 'Mechanical Keyboard',
  'category': 'electronics',
  'price': '149'},
 {'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GH',
  'name': 'Gaming Laptop Pro',
  'category': 'electronics',
  'price': '1299'},
 {'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GJ',
  'name': 'Budget Laptop Basic',
  'category': 'electronics',
  'price': '499'},
 {'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GR',
  'name': 'USB-C Hub',
  'category': 'electronics',
  'price': '49'}]

In [7]:
# YOUR CODE HERE - Method 2: SQLQuery
# Hint: SELECT ... FROM products_exercise WHERE category = 'electronics'
sql_query = SQLQuery(f"""
    SELECT name, category, price
    FROM {index.name}
    WHERE category = 'electronics'
""")
index.query(sql_query)

[{'name': 'Wireless Mouse', 'category': 'electronics', 'price': '35'},
 {'name': 'Mechanical Keyboard', 'category': 'electronics', 'price': '149'},
 {'name': 'Gaming Laptop Pro', 'category': 'electronics', 'price': '1299'},
 {'name': 'Budget Laptop Basic', 'category': 'electronics', 'price': '499'},
 {'name': 'USB-C Hub', 'category': 'electronics', 'price': '49'}]

In [9]:
# YOUR CODE HERE - Method 3: Raw FT.SEARCH
# Hint: client.execute_command("FT.SEARCH", index_name, "@category:{electronics}", ...)
client.execute_command("FT.Search", index.name, "@category:{electronics}")

[5,
 b'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GK',
 [b'name',
  b'Wireless Mouse',
  b'description',
  b'Ergonomic wireless mouse',
  b'category',
  b'electronics',
  b'brand',
  b'TechBrand',
  b'price',
  b'35',
  b'stock',
  b'200',
  b'rating',
  b'4.3',
  b'embedding',
  b'333?\x9a\x99\x99>\xcd\xcc\xcc>\x00\x00\x00?'],
 b'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GQ',
 [b'name',
  b'Mechanical Keyboard',
  b'description',
  b'Premium mechanical keyboard with RGB',
  b'category',
  b'electronics',
  b'brand',
  b'KeyMaster',
  b'price',
  b'149',
  b'stock',
  b'80',
  b'rating',
  b'4.6',
  b'embedding',
  b'\x9a\x99\x19?\xcd\xcc\xcc>\x00\x00\x00?\x9a\x99\x19?'],
 b'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GH',
 [b'name',
  b'Gaming Laptop Pro',
  b'description',
  b'High-performance laptop for gaming',
  b'category',
  b'electronics',
  b'brand',
  b'TechBrand',
  b'price',
  b'1299',
  b'stock',
  b'15',
  b'rating',
  b'4.7',
  b'embedding',
  b'fff?\xcd\xcc\xcc=\xcd\xccL>\x9a

### Solution: Exercise 1

In [ ]:
# Method 1: RedisVL Python API
filter_expr = Tag("category") == "electronics"
query = FilterQuery(filter_expression=filter_expr, return_fields=["name", "category", "price"], num_results=10)
results_api = index.query(query)
print("=== Method 1: RedisVL Python API ===")
for r in results_api:
    print(f"  {r['name']} - ${r['price']}")

In [ ]:
# Method 2: RedisVL SQL
sql_query = SQLQuery(f"""
    SELECT name, category, price
    FROM {index.name}
    WHERE category = 'electronics'
""")
results_sql = index.query(sql_query)
print("=== Method 2: RedisVL SQL ===")
for r in results_sql:
    print(f"  {r['name']} - ${r['price']}")

# Show the translated Redis command
redis_cmd = sql_query.redis_query_string(redis_client=client)
print(f"\nTranslated Redis command: {redis_cmd}")

In [ ]:
# Method 3: Raw Redis FT.SEARCH
raw_results = client.execute_command("FT.SEARCH", index.name, "@category:{electronics}", "RETURN", "3", "name", "category", "price", "LIMIT", "0", "10")
print("=== Method 3: Raw FT.SEARCH ===")
total = raw_results[0]
print(f"Total matches: {total}")
for i in range(1, len(raw_results), 2):
    if i + 1 < len(raw_results):
        fields = raw_results[i + 1]
        field_dict = {fields[j].decode(): fields[j+1].decode() for j in range(0, len(fields), 2)}
        print(f"  {field_dict.get('name', 'N/A')} - ${field_dict.get('price', 'N/A')}")


---
## Exercise 2: Numeric Range Queries

**Goal:** Find all products with price between $40 and $100.

### Do It Yourself

**Documentation:**
- [RedisVL Numeric Filters](https://docs.redisvl.com/en/latest/api/query.html#redisvl.query.filter.Num)
- [Redis Numeric Range Queries](https://redis.io/docs/latest/develop/ai/search-and-query/query/range/)

In [14]:
# YOUR CODE HERE - Method 1: RedisVL Python API
# Hint: Use Num("price").between(40, 100) with FilterQuery
q = FilterQuery(
    filter_expression = Num("price").between(40,100),
    return_fields=["name", "price"],
    num_results=10
)
index.query(q)

[{'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GM',
  'name': 'Python Programming Guide',
  'price': '45'},
 {'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GN',
  'name': 'Redis in Action',
  'price': '55'},
 {'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GP',
  'name': 'Data Science Handbook',
  'price': '65'},
 {'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GR',
  'name': 'USB-C Hub',
  'price': '49'},
 {'id': 'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GT',
  'name': 'Monitor Stand',
  'price': '79'}]

In [17]:
# YOUR CODE HERE - Method 2: SQLQuery
# Hint: SELECT ... WHERE price BETWEEN 40 AND 100
sql_query = SQLQuery(f"""
    SELECT  name, price from {index.name} where price between 40 and 100
""")

index.query(sql_query)


[{'name': 'Python Programming Guide', 'price': '45'},
 {'name': 'Redis in Action', 'price': '55'},
 {'name': 'Data Science Handbook', 'price': '65'},
 {'name': 'USB-C Hub', 'price': '49'},
 {'name': 'Monitor Stand', 'price': '79'}]

In [19]:
# YOUR CODE HERE - Method 3: Raw FT.SEARCH
# Hint: @price:[40 100]
client.execute_command("FT.SEARCH", index.name, "@price:[40 100]", "RETURN", "2", "name", "price", "LIMIT", "0", "10")

[5,
 b'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GM',
 [b'name', b'Python Programming Guide', b'price', b'45'],
 b'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GN',
 [b'name', b'Redis in Action', b'price', b'55'],
 b'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GP',
 [b'name', b'Data Science Handbook', b'price', b'65'],
 b'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GR',
 [b'name', b'USB-C Hub', b'price', b'49'],
 b'product_exercise:01KGR04DCSDX9D0KSD7GAYC1GT',
 [b'name', b'Monitor Stand', b'price', b'79']]

### Solution: Exercise 2

In [ ]:
# Method 1: RedisVL Python API
filter_expr = Num("price").between(40, 100)
query = FilterQuery(filter_expression=filter_expr, return_fields=["name", "price"], num_results=10)
results_api = index.query(query)
print("=== Method 1: RedisVL Python API ===")
for r in results_api:
    print(f"  {r['name']} - ${r['price']}")

In [ ]:
# Method 2: RedisVL SQL
sql_query = SQLQuery(f"""
    SELECT name, price
    FROM {index.name}
    WHERE price BETWEEN 40 AND 100
""")
results_sql = index.query(sql_query)
print("=== Method 2: RedisVL SQL ===")
for r in results_sql:
    print(f"  {r['name']} - ${r['price']}")

redis_cmd = sql_query.redis_query_string(redis_client=client)
print(f"\nTranslated Redis command: {redis_cmd}")

In [ ]:
# Method 3: Raw Redis FT.SEARCH
raw_results = client.execute_command("FT.SEARCH", index.name, "@price:[40 100]", "RETURN", "2", "name", "price", "LIMIT", "0", "10")
print("=== Method 3: Raw FT.SEARCH ===")
for i in range(1, len(raw_results), 2):
    if i + 1 < len(raw_results):
        fields = raw_results[i + 1]
        field_dict = {fields[j].decode(): fields[j+1].decode() for j in range(0, len(fields), 2)}
        print(f"  {field_dict.get('name', 'N/A')} - ${field_dict.get('price', 'N/A')}")


---
## Exercise 3: Combined Filters (AND/OR)

**Goal:** Find electronics products under $100.

### Do It Yourself

**Documentation:**
- [RedisVL Filter Expressions](https://docs.redisvl.com/en/latest/api/query.html#filter-expressions)
- [Redis Combined Queries](https://redis.io/docs/latest/develop/ai/search-and-query/query/)

In [ ]:
# YOUR CODE HERE - Method 1: RedisVL Python API
# Hint: Combine filters with & operator: (Tag("category") == "electronics") & (Num("price") < 100)


In [ ]:
# YOUR CODE HERE - Method 2: SQLQuery
# Hint: WHERE category = 'electronics' AND price < 100


In [ ]:
# YOUR CODE HERE - Method 3: Raw FT.SEARCH
# Hint: (@category:{electronics} @price:[-inf 100])


### Solution: Exercise 3

In [ ]:
# Method 1: RedisVL Python API
filter_expr = (Tag("category") == "electronics") & (Num("price") < 100)
query = FilterQuery(filter_expression=filter_expr, return_fields=["name", "category", "price"], num_results=10)
results_api = index.query(query)
print("=== Method 1: RedisVL Python API ===")
for r in results_api:
    print(f"  {r['name']} ({r['category']}) - ${r['price']}")

In [ ]:
# Method 2: RedisVL SQL
sql_query = SQLQuery(f"""
    SELECT name, category, price
    FROM {index.name}
    WHERE category = 'electronics' AND price < 100
""")
results_sql = index.query(sql_query)
print("=== Method 2: RedisVL SQL ===")
for r in results_sql:
    print(f"  {r['name']} ({r['category']}) - ${r['price']}")

redis_cmd = sql_query.redis_query_string(redis_client=client)
print(f"\nTranslated Redis command: {redis_cmd}")

In [ ]:
# Method 3: Raw Redis FT.SEARCH
raw_results = client.execute_command("FT.SEARCH", index.name, "(@category:{electronics} @price:[-inf (100])", "RETURN", "3", "name", "category", "price", "LIMIT", "0", "10")
print("=== Method 3: Raw FT.SEARCH ===")
for i in range(1, len(raw_results), 2):
    if i + 1 < len(raw_results):
        fields = raw_results[i + 1]
        field_dict = {fields[j].decode(): fields[j+1].decode() for j in range(0, len(fields), 2)}
        print(f"  {field_dict.get('name', 'N/A')} ({field_dict.get('category', 'N/A')}) - ${field_dict.get('price', 'N/A')}")


---
## Exercise 4: Text Search

**Goal:** Find products with "laptop" in the name.

### Do It Yourself

**Documentation:**
- [RedisVL Text Filters](https://docs.redisvl.com/en/latest/api/query.html#redisvl.query.filter.Text)
- [Redis Full-Text Search](https://redis.io/docs/latest/develop/ai/search-and-query/query/full-text/)

In [ ]:
# YOUR CODE HERE - Method 1: RedisVL Python API
# Hint: Use Text("name") % "laptop" with FilterQuery


In [ ]:
# YOUR CODE HERE - Method 2: SQLQuery
# Hint: WHERE name = 'laptop'


In [ ]:
# YOUR CODE HERE - Method 3: Raw FT.SEARCH
# Hint: @name:laptop


### Solution: Exercise 4

In [ ]:
# Method 1: RedisVL Python API
filter_expr = Text("name") % "laptop"
query = FilterQuery(filter_expression=filter_expr, return_fields=["name", "description", "price"], num_results=10)
results_api = index.query(query)
print("=== Method 1: RedisVL Python API ===")
for r in results_api:
    print(f"  {r['name']} - ${r['price']}")

In [ ]:
# Method 2: RedisVL SQL
sql_query = SQLQuery(f"""
    SELECT name, description, price
    FROM {index.name}
    WHERE name = 'laptop'
""")
results_sql = index.query(sql_query)
print("=== Method 2: RedisVL SQL ===")
for r in results_sql:
    print(f"  {r['name']} - ${r['price']}")

redis_cmd = sql_query.redis_query_string(redis_client=client)
print(f"\nTranslated Redis command: {redis_cmd}")

In [ ]:
# Method 3: Raw Redis FT.SEARCH
raw_results = client.execute_command("FT.SEARCH", index.name, "@name:laptop", "RETURN", "3", "name", "description", "price", "LIMIT", "0", "10")
print("=== Method 3: Raw FT.SEARCH ===")
for i in range(1, len(raw_results), 2):
    if i + 1 < len(raw_results):
        fields = raw_results[i + 1]
        field_dict = {fields[j].decode(): fields[j+1].decode() for j in range(0, len(fields), 2)}
        print(f"  {field_dict.get('name', 'N/A')} - ${field_dict.get('price', 'N/A')}")



---
## Exercise 5: Vector Similarity Search

**Goal:** Find products most similar to a query vector (simulating a semantic search).

### Do It Yourself

**Documentation:**
- [RedisVL VectorQuery](https://docs.redisvl.com/en/latest/api/query.html#vectorquery)
- [Redis Vector Search](https://redis.io/docs/latest/develop/ai/search-and-query/vectors/)

In [ ]:
# YOUR CODE HERE - Method 1: RedisVL Python API
# Hint: Use VectorQuery with a query vector


In [ ]:
# YOUR CODE HERE - Method 2: SQLQuery
# Hint: SELECT ... ORDER BY cosine_distance(embedding, <vector>) LIMIT k


In [ ]:
# YOUR CODE HERE - Method 3: Raw FT.SEARCH
# Hint: FT.SEARCH with KNN and BLOB parameter


### Solution: Exercise 5

In [21]:
# Query vector (similar to electronics products)
query_vector = np.array([0.85, 0.15, 0.25, 0.35], dtype=np.float32)

# Method 1: RedisVL Python API
vector_query = VectorQuery(
    vector=query_vector,
    vector_field_name="embedding",
    return_fields=["name", "category", "price"],
    num_results=3
)
results_api = index.query(vector_query)
print("=== Method 1: RedisVL Python API ===")
for r in results_api:
    print(f"  {r['name']} ({r['category']}) - distance: {r.get('vector_distance', 'N/A')}")

=== Method 1: RedisVL Python API ===
  Gaming Laptop Pro (electronics) - distance: 0.00526285171509
  Budget Laptop Basic (electronics) - distance: 0.00537633895874
  Wireless Mouse (electronics) - distance: 0.0464093089104


In [26]:
# Method 2: RedisVL SQL
# Note: sql-redis uses cosine_distance() function for vector search
vector_bytes = query_vector.tobytes()
sql_query = SQLQuery(f"""
    SELECT name, category, price
    FROM {index.name}
    ORDER BY cosine_distance(embedding, :vector)
    LIMIT 3
""",
params={"vector": vector_bytes}
                     )


results_sql = index.query(sql_query)
print("=== Method 2: RedisVL SQL ===")
for r in results_sql:
    print(f"  {r['name']} ({r['category']})")

redis_cmd = sql_query.redis_query_string(redis_client=client)
print(f"\nTranslated Redis command: {redis_cmd[:100]}...")



=== Method 2: RedisVL SQL ===
  Wireless Mouse (electronics)
  Monitor Stand (accessories)
  Redis in Action (books)

Translated Redis command: FT.SEARCH products_exercise "*" RETURN 3 name category price LIMIT 0 3...


In [ ]:
# Method 3: Raw Redis FT.SEARCH with KNN
import struct
vector_blob = struct.pack(f'{len(query_vector)}f', *query_vector)
raw_results = client.execute_command(
    "FT.SEARCH", index.name,
    "*=>[KNN 3 @embedding $vec AS vector_distance]",
    "PARAMS", "2", "vec", vector_blob,
    "RETURN", "4", "name", "category", "price", "vector_distance",
    "SORTBY", "vector_distance", "ASC",
    "DIALECT", "2"
)
print("=== Method 3: Raw FT.SEARCH ===")
for i in range(1, len(raw_results), 2):
    if i + 1 < len(raw_results):
        fields = raw_results[i + 1]
        field_dict = {fields[j].decode() if isinstance(fields[j], bytes) else fields[j]:
                      fields[j+1].decode() if isinstance(fields[j+1], bytes) else fields[j+1]
                      for j in range(0, len(fields), 2)}
        print(f"  {field_dict.get('name', 'N/A')} ({field_dict.get('category', 'N/A')}) - distance: {field_dict.get('vector_distance', 'N/A')}")



---
## Exercise 6: Aggregations (COUNT, GROUP BY, AVG)

**Goal:** Count products by category and calculate average prices.

### Do It Yourself

**Documentation:**
- [RedisVL CountQuery](https://docs.redisvl.com/en/latest/api/query.html#countquery)
- [Redis Aggregations](https://redis.io/docs/latest/develop/ai/search-and-query/advanced-concepts/aggregations/)

In [ ]:
# YOUR CODE HERE - Method 1: RedisVL Python API
# Hint: Use CountQuery for counting


In [ ]:
# YOUR CODE HERE - Method 2: SQLQuery
# Hint: SELECT category, COUNT(*), AVG(price) FROM ... GROUP BY category


In [ ]:
# YOUR CODE HERE - Method 3: Raw FT.AGGREGATE
# Hint: FT.AGGREGATE with GROUPBY and REDUCE


### Solution: Exercise 6

In [27]:
# Method 1: RedisVL Python API - Count total products
count_query = CountQuery(filter_expression=Tag("category") == "electronics")
count_result = index.query(count_query)
print("=== Method 1: RedisVL Python API ===")
print(f"  Electronics products count: {count_result}")

# Count for each category
for cat in ["electronics", "books", "accessories"]:
    count_query = CountQuery(filter_expression=Tag("category") == cat)
    count = index.query(count_query)
    print(f"  {cat}: {count} products")

=== Method 1: RedisVL Python API ===
  Electronics products count: 5
  electronics: 5 products
  books: 3 products
  accessories: 2 products


In [28]:
# Method 2: RedisVL SQL - Group by with aggregations
sql_query = SQLQuery(f"""
    SELECT category, COUNT(*) as count, AVG(price) as avg_price
    FROM {index.name}
    GROUP BY category
""")
results_sql = index.query(sql_query)
print("=== Method 2: RedisVL SQL ===")
for r in results_sql:
    print(f"  {r['category']}: {r['count']} products, avg price: ${float(r['avg_price']):.2f}")

redis_cmd = sql_query.redis_query_string(redis_client=client)
print(f"\nTranslated Redis command: {redis_cmd}")

=== Method 2: RedisVL SQL ===
  books: 3 products, avg price: $55.00
  accessories: 2 products, avg price: $59.00
  electronics: 5 products, avg price: $406.20

Translated Redis command: FT.AGGREGATE products_exercise "*" LOAD 2 category price GROUPBY 1 @category REDUCE COUNT 0 AS count REDUCE AVG 1 @price AS avg_price


In [29]:
# Method 3: Raw Redis FT.AGGREGATE
raw_results = client.execute_command(
    "FT.AGGREGATE", index.name, "*",
    "GROUPBY", "1", "@category",
    "REDUCE", "COUNT", "0", "AS", "count",
    "REDUCE", "AVG", "1", "@price", "AS", "avg_price"
)
print("=== Method 3: Raw FT.AGGREGATE ===")
for i in range(1, len(raw_results)):
    row = raw_results[i]
    row_dict = {row[j].decode() if isinstance(row[j], bytes) else row[j]:
                row[j+1].decode() if isinstance(row[j+1], bytes) else row[j+1]
                for j in range(0, len(row), 2)}
    cat = row_dict.get('category', 'N/A')
    count = row_dict.get('count', 'N/A')
    avg_price = float(row_dict.get('avg_price', 0))
    print(f"  {cat}: {count} products, avg price: ${avg_price:.2f}")



=== Method 3: Raw FT.AGGREGATE ===
  books: 3 products, avg price: $55.00
  accessories: 2 products, avg price: $59.00
  electronics: 5 products, avg price: $406.20


---
## Comparison Summary

| Approach | Pros | Cons | Best For |
|----------|------|------|----------|
| **RedisVL Python API** | Type-safe, IDE autocomplete, Pythonic | Learning curve for filter expressions | Production applications, complex queries |
| **RedisVL SQL** | Familiar SQL syntax, easy migration | Limited to SQL capabilities | SQL developers, quick prototyping |
| **Raw FT.SEARCH** | Full control, all Redis features | Verbose, error-prone | Advanced use cases, debugging |

### Key Takeaways

1. **SQLQuery** is great for developers familiar with SQL who want to quickly query Redis
2. **RedisVL Python API** provides the best developer experience with type safety
3. **Raw FT.SEARCH** gives you full control but requires deep Redis knowledge
4. All three methods can achieve the same results - choose based on your team's expertise


---
## Cleanup

In [ ]:
# Delete the index and clean up
index.delete(drop=True)
print(f"Deleted index: {index.name}")
print("Cleanup complete!")

---
## Next Steps

- Explore more complex queries in the [RedisVL documentation](https://docs.redisvl.com)
- Try the [sql-redis package](https://pypi.org/project/sql-redis/) for more SQL features
- Check out other RedisVL features like SemanticCache and SemanticRouter